# Stock Manager — Full Training & Backtesting Pipeline

End-to-end notebook: **data ingestion → EDA → model training → backtest comparison**, with full MLflow traceability.

| # | Section | Description |
|---|---------|-------------|
| 0 | Environment Setup | Install deps, imports, path setup |
| 1 | Configuration | All tuneable parameters in one place |
| 2 | Data Ingestion | Download & label OHLCV data |
| 3 | Exploratory Data Analysis | Data quality, price charts, return distributions, correlations |
| 4 | Qlib Dataset Build | Optional Qlib provider format |
| 5 | MLflow Setup | Configure experiment tracking |
| 6 | Model Training | LightGBM · MASTER · StockMixer (each logged to MLflow) |
| 7 | Post-training Visualisation | Feature importance, rolling IC, prediction scatter |
| 8 | Backtesting | Long-short strategy comparison across models & cost assumptions |
| 9 | Post-backtest Visualisation | Equity curves, drawdowns, Sharpe comparison, cost sensitivity |

> **Tip:** Set `STOCK_MANAGER_DATA_DIR`, `STOCK_MANAGER_MODEL_DIR`, and `STOCK_MANAGER_REPORT_DIR` as env vars **or** edit the paths in Section 1 before running.
> Run cells **top-to-bottom**. Sections 6–9 can be re-run independently once earlier outputs exist on disk.

## 1  Environment Setup & Imports

In [ ]:
import subprocess, sys, os
from pathlib import Path

# ── Detect environment ────────────────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    _ON_COLAB = True
except ImportError:
    _ON_COLAB = False

print(f"Installing PyPI packages …  (Colab={_ON_COLAB})")

# ── Heavy deps first (separate so pip can parallelise) ────────────────────────
_heavy = [
    "torch>=2.1",
    "lightgbm>=4.0",
    "pyqlib>=0.9",
    "mlflow>=2.10",
    "optuna>=3.4",
    "seaborn>=0.13",
]
subprocess.run([sys.executable, "-m", "pip", "install", *_heavy, "--quiet"], check=True)
print("✓ Heavy packages done")

# ── Other PyPI deps ───────────────────────────────────────────────────────────
_others = [
    "pyarrow>=14.0", "yfinance>=0.2.40", "alpaca-py>=0.28",
    "fredapi>=0.5", "scikit-learn>=1.3", "matplotlib>=3.8",
    "tqdm>=4.66", "pandas>=2.0", "numpy>=1.24",
]
subprocess.run([sys.executable, "-m", "pip", "install", *_others, "--quiet"], check=True)
print("✓ PyPI packages done")

# ── Install the stock-manager package ────────────────────────────────────────
if _ON_COLAB:
    # Install from GitHub (subdirectory = Training where pyproject.toml lives)
    _git_url = (
        "stock-manager[all] @ "
        "git+https://github.com/SiddarthaKoppaka/stocko.git#subdirectory=Training"
    )
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", _git_url, "--quiet"],
            check=True,
        )
        print("✓ stock-manager installed from GitHub")
    except subprocess.CalledProcessError:
        print(
            "⚠  GitHub install failed (private repo or not yet pushed).\n"
            "   Falling back to sys.path from Drive — see config cell (Section 1)."
        )
else:
    # Local dev: editable install from repo root (one level up from notebooks/)
    _repo_root = Path("..").resolve()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", str(_repo_root), "--quiet"],
        check=True,
    )
    print(f"✓ stock-manager installed (editable) from {_repo_root}")


Installing PyPI packages …  (Colab=True)
✓ PyPI packages done
ℹ  Colab detected — local package will be installed after Drive is mounted in Section 1


In [6]:
import os, sys, json, warnings, pickle
from pathlib import Path

# ── ensure repo src/ is on the path (local dev; no-op on Colab after pip install) ──
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# ── standard / third-party packages (always available after cell 1) ───────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import mlflow

warnings.filterwarnings("ignore")

# ── plot defaults ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (13, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
sns.set_theme(style="whitegrid", palette="muted")

print(f"✓ stdlib/third-party imports OK  |  Python {sys.version.split()[0]}  |  MLflow {mlflow.__version__}")
print("ℹ  Project imports (stock_manager.*) will load in the cell below Section 1 config.")


✓ stdlib/third-party imports OK  |  Python 3.12.13  |  MLflow 3.12.0
ℹ  Project imports (stock_manager.*) will load in the cell below Section 1 config.


## 1  Configuration & Parameters

All tuneable parameters live here. Edit this cell before running the rest of the pipeline.

In [ ]:
import subprocess, sys, os
from pathlib import Path

# ── Detect environment ────────────────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    _ON_COLAB = True
except ImportError:
    _ON_COLAB = False

# ── Colab: mount Drive, then locate and register the local package ───────────
_colab_src_path: str | None = None

if _ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Candidate Drive paths (most-specific first)
    _DRIVE_CANDIDATES = [
        Path("/content/drive/MyDrive/Stock_manager/src"),
        Path("/content/drive/MyDrive/Stock_manager/Training/src"),
        Path("/content/drive/MyDrive/stock_manager/src"),
        Path("/content/drive/MyDrive/stock_manager/Training/src"),
    ]
    _src_dir: Path | None = None
    for _c in _DRIVE_CANDIDATES:
        if (_c / "stock_manager" / "__init__.py").exists():
            _src_dir = _c
            break

    if _src_dir is None:
        # Wider search — expensive but only done once
        _roots = [
            Path("/content/drive/MyDrive/Stock_manager"),
            Path("/content/drive/MyDrive/stock_manager"),
            Path("/content/drive/MyDrive"),
        ]
        for _r in _roots:
            if _r.exists():
                _hits = list(_r.rglob("stock_manager/__init__.py"))
                if _hits:
                    _src_dir = _hits[0].parent.parent   # src/
                    break

    if _src_dir is not None:
        _colab_src_path = str(_src_dir)
        if _colab_src_path not in sys.path:
            sys.path.insert(0, _colab_src_path)
        print(f"✓ stock_manager found at: {_colab_src_path}")
    else:
        print(
            "\n⚠  stock_manager source NOT found on Google Drive.\n"
            "   Please upload the 'src/' folder (from your local project) to Drive.\n"
            "   Expected location: MyDrive/Stock_manager/src/stock_manager/\n"
            "   Then re-run this cell.\n"
        )

    # Infer _PROJECT_ROOT from where we found src/ (or fall back to drive root)
    _DRIVE_ROOT = Path("/content/drive/MyDrive/Stock_manager")
    _PROJECT_ROOT = _src_dir.parent if _src_dir is not None else _DRIVE_ROOT
else:
    _PROJECT_ROOT = REPO_ROOT   # set in the imports cell above
    print("Local environment — using REPO_ROOT")

print(f"Environment    : {'Colab' if _ON_COLAB else 'local'}")
print(f"_PROJECT_ROOT  : {_PROJECT_ROOT}")

# ── Artifact directories ──────────────────────────────────────────────────────
DATA_DIR   = Path(os.environ.get("STOCK_MANAGER_DATA_DIR",   str(_PROJECT_ROOT / "data")))
MODEL_DIR  = Path(os.environ.get("STOCK_MANAGER_MODEL_DIR",  str(_PROJECT_ROOT / "models")))
REPORT_DIR = Path(os.environ.get("STOCK_MANAGER_REPORT_DIR", str(_PROJECT_ROOT / "reports")))

for _p in [DATA_DIR, MODEL_DIR, REPORT_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

for _p in [
    DATA_DIR / "raw", DATA_DIR / "processed", DATA_DIR / "qlib",
    MODEL_DIR / "lightgbm", MODEL_DIR / "master", MODEL_DIR / "stockmixer",
    REPORT_DIR / "backtests", REPORT_DIR / "plots",
]:
    _p.mkdir(parents=True, exist_ok=True)

os.environ["STOCK_MANAGER_DATA_DIR"]   = str(DATA_DIR)
os.environ["STOCK_MANAGER_MODEL_DIR"]  = str(MODEL_DIR)
os.environ["STOCK_MANAGER_REPORT_DIR"] = str(REPORT_DIR)

# ── API keys (Colab secrets or pre-set env vars) ──────────────────────────────
if _ON_COLAB:
    try:
        from google.colab import userdata
        for _key in ("ALPACA_API_KEY", "ALPACA_SECRET_KEY", "FRED_API_KEY"):
            try:
                os.environ[_key] = userdata.get(_key)
            except Exception:
                pass
        print("✓ Secrets loaded from Colab")
    except Exception:
        pass

# ── Date window (rolling 10 years) ───────────────────────────────────────────
TODAY      = pd.Timestamp.now(tz="UTC").tz_convert(None).normalize()
START_DATE = (TODAY - pd.DateOffset(years=10)).strftime("%Y-%m-%d")
END_DATE   = TODAY.strftime("%Y-%m-%d")

INGEST_LABEL_COL  = "label_5d"
PROCESSED_PATH    = DATA_DIR / "processed/ohlcv_labeled.parquet"
QLIB_PROVIDER_URI = DATA_DIR / "qlib/us_equities"
ALPHA158_PATH     = QLIB_PROVIDER_URI / "alpha158.parquet"
RUNTIME_CFG       = {"show_progress": True}

# ── Data parameters ───────────────────────────────────────────────────────────
INGEST_CFG = {
    "universe": {"name": "sp500_current", "source": "sp500_auto"},
    "data": {
        "start": START_DATE,
        "end": END_DATE,
        "providers": ["yfinance"],
        "label_horizon": 5,
    },
    "fred": {"enabled": False, "series": ["DGS10", "VIXCLS"]},
    "paths": {
        "raw_dir": str(DATA_DIR / "raw"),
        "processed_dir": str(DATA_DIR / "processed"),
    },
}

# ── Chronological splits ──────────────────────────────────────────────────────
TRAIN_END = (TODAY - pd.DateOffset(years=3)).strftime("%Y-%m-%d")
VALID_END = (TODAY - pd.DateOffset(years=2)).strftime("%Y-%m-%d")
SPLITS = {"train_end": TRAIN_END, "valid_end": VALID_END, "test_end": END_DATE}

# ── Model configs ─────────────────────────────────────────────────────────────
_BASE = {
    "splits":  SPLITS,
    "paths":   {"model_dir": str(MODEL_DIR)},
    "runtime": RUNTIME_CFG,
}

LGBM_CFG = {
    **_BASE,
    "data": {"processed_path": str(ALPHA158_PATH), "label_column": "LABEL0"},
    "model": {
        "type": "lightgbm_alpha158",
        "params": {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "early_stopping_rounds": 20,
            "num_leaves": 63,
            "n_jobs": -1,
        },
    },
    "paths": {"model_dir": str(MODEL_DIR), "report_dir": str(REPORT_DIR / "lightgbm_alpha158")},
}
MASTER_CFG = {
    **_BASE,
    "data": {
        "processed_path": str(ALPHA158_PATH),
        "market_source_path": str(PROCESSED_PATH),
        "label_column": "LABEL0",
    },
    "model": {
        "type": "master",
        "params": {
            "lookback_window": 8,
            "d_model": 256,
            "t_nhead": 4,
            "s_nhead": 2,
            "t_dropout_rate": 0.5,
            "s_dropout_rate": 0.5,
            "beta": 2.0,
            "n_epochs": 40,
            "lr": 1.0e-04,
            "optimizer": "adamw",
            "weight_decay": 1e-2,
            "grad_clip": 3.0,
            "early_stopping_patience": 5,
            "early_stopping_min_delta": 1.0e-04,
            "seed": 0,
        },
    },
    "paths": {"model_dir": str(MODEL_DIR), "report_dir": str(REPORT_DIR / "master")},
}
STOCKMIXER_CFG = {
    **_BASE,
    "data": {"processed_path": str(PROCESSED_PATH)},
    "model": {
        "type": "stockmixer",
        "params": {
            "lookback_length": 16,
            "steps": 1,
            "epochs": 100,
            "learning_rate": 0.001,
            "optimizer": "adam",
            "weight_decay": 0.0,
            "grad_clip": 5.0,
            "normalization_window": 20,
            "alpha": 0.1,
            "market_hidden_dim": 20,
            "scale_factor": 3,
            "seed": 12345678,
        },
    },
    "paths": {"model_dir": str(MODEL_DIR), "report_dir": str(REPORT_DIR / "stockmixer")},
}

# ── Backtest comparison ───────────────────────────────────────────────────────
BACKTEST_CFG = {
    "report_dir": str(REPORT_DIR / "comparison"),
    "cost_bps": [0, 5, 10],
    "models": [
        {"name": "lightgbm_alpha158", "predictions_path": str(REPORT_DIR / "lightgbm_alpha158/lightgbm_alpha158_predictions.parquet")},
        {"name": "master",            "predictions_path": str(REPORT_DIR / "master/master_predictions.parquet")},
        {"name": "stockmixer",        "predictions_path": str(REPORT_DIR / "stockmixer/stockmixer_predictions.parquet")},
    ],
}

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = str(DATA_DIR / "mlruns")
EXPERIMENT_NAME     = "stock_manager_pipeline"

print(f"\nDATA_DIR       : {DATA_DIR}")
print(f"MODEL_DIR      : {MODEL_DIR}")
print(f"REPORT_DIR     : {REPORT_DIR}")
print(f"PROCESSED_PATH : {PROCESSED_PATH}")
print(f"ALPHA158_PATH  : {ALPHA158_PATH}")
print(f"MLflow URI     : {MLFLOW_TRACKING_URI}")
print(f"Universe       : {INGEST_CFG['universe']['source']}")
print(f"Dates          : {START_DATE} -> {END_DATE}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠  pip install -e failed — falling back to sys.path injection
   stderr: ERROR: file:///content/drive/MyDrive/Stock_manager does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

✓ stock_manager source added to sys.path: /content/drive/MyDrive/Stock_manager/src
Environment : Colab
✓ Secrets loaded from Colab
DATA_DIR       : /content/drive/MyDrive/Stock_manager/data
MODEL_DIR      : /content/drive/MyDrive/Stock_manager/models
REPORT_DIR     : /content/drive/MyDrive/Stock_manager/reports
PROCESSED_PATH : /content/drive/MyDrive/Stock_manager/data/processed/ohlcv_labeled.parquet
ALPHA158_PATH  : /content/drive/MyDrive/Stock_manager/data/qlib/us_equities/alpha158.parquet
Progress bars  : True
MLflow URI     : /content/drive/MyDrive/Stock_manager/data/mlruns
Universe       : sp500_auto (sp500_current)
Dates          : 2016-05-13 ->

In [ ]:
# ── Project imports ───────────────────────────────────────────────────────────
# Must run AFTER the config cell (Section 1) so Drive is mounted on Colab.
try:
    from stock_manager.data.ingestion import ingest_market_data
    from stock_manager.qlib.converter import build_qlib_dataset
    from stock_manager.models.base import train_from_config
    from stock_manager.backtest.compare import compare_models
    from stock_manager.backtest.metrics import long_short_returns
    print("✓ project imports OK")
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "\n\nstock_manager package not found.\n\n"
        "On Colab: upload the 'src/' folder from your local project to Google Drive at:\n"
        "  MyDrive/Stock_manager/src/\n"
        "so that 'MyDrive/Stock_manager/src/stock_manager/__init__.py' exists,\n"
        "then re-run the Section 1 config cell, then re-run this cell.\n\n"
        "Local: ensure you ran  pip install -e ..[all]  from the repo root."
    )


Drive root contents:
  [F]  01_ingestion (1).ipynb
  [F]  01_ingestion.ipynb
  [F]  02_model_training.ipynb
  [F]  03_paper_trading.ipynb
  [F]  README.txt
  [D]  data
  [D]  logs
  [D]  models
  [D]  processed
  [D]  reports

Searching for stock_manager/__init__.py ...
  Found: []


ModuleNotFoundError: No module named 'stock_manager'

## 2  Data Ingestion

Download OHLCV bars via yfinance (or Alpaca if credentials are set), apply the 5-day forward-return label, and persist raw + processed parquet files.

In [ ]:
print("Ingesting market data …")
ingest_outputs = ingest_market_data(INGEST_CFG)

print(f"  Raw OHLCV  → {ingest_outputs['raw']}")
print(f"  Labeled    → {ingest_outputs['processed']}")
if "fred" in ingest_outputs:
    print(f"  FRED macro → {ingest_outputs['fred']}")

# Load frames for downstream use
df_raw       = pd.read_parquet(ingest_outputs["raw"])
df_processed = pd.read_parquet(ingest_outputs["processed"])

print(f"\nRaw rows   : {len(df_raw):,}")
print(f"Labeled rows: {len(df_processed):,}")
print(f"Date range : {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
print(f"Tickers    : {sorted(df_raw['ticker'].unique())}")

## 3  Exploratory Data Analysis

Understand the data before training: quality report, price overview, return distribution, volume trends, and ticker correlations.

In [ ]:
# ── 3.1  Data quality report ─────────────────────────────────────────────────
with open(ingest_outputs["report"]) as f:
    report = json.load(f)

print("=" * 55)
print(" DATA QUALITY REPORT")
print("=" * 55)
print(f"  Rows              : {report['row_count']:,}")
print(f"  Tickers           : {report['ticker_count']}")
print(f"  Duplicate rows    : {report['duplicate_rows']}")
print(f"  Provider used     : {report.get('provider', 'N/A')}")
print(f"  Tickers requested : {report.get('tickers_requested', 'N/A')}")
print("\nMissing values per column:")
for col, cnt in report["missing_by_column"].items():
    pct = cnt / report["row_count"] * 100 if report["row_count"] else 0
    bar = "█" * int(pct * 2)
    status = "✓" if cnt == 0 else "⚠"
    print(f"  {status} {col:12s}: {cnt:5d}  ({pct:.2f}%)  {bar}")

In [ ]:
# ── 3.2  Normalised closing price overview ───────────────────────────────────
top_tickers = (
    df_raw.groupby("ticker")["volume"]
    .sum()
    .nlargest(6)
    .index.tolist()
)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for ax, ticker in zip(axes, top_tickers):
    grp = df_raw[df_raw["ticker"] == ticker].set_index("date").sort_index()["close"]
    norm = grp / grp.iloc[0] * 100
    ax.plot(norm, linewidth=1.4, color="#4C72B0")
    ax.set_title(ticker, fontweight="bold", fontsize=12)
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f"))
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Normalised Close Price (rebased = 100) — top 6 tickers by volume",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3  Forward-return label distribution ───────────────────────────────────
label_col = INGEST_LABEL_COL
if label_col not in df_processed.columns:
    available = ", ".join(df_processed.columns.tolist())
    raise ValueError(
        f"Section 3 uses df_processed from ingestion, so it expects '{INGEST_LABEL_COL}' on that frame. "
        f"Available columns: {available}. If you want to inspect LABEL0, build Alpha158 in Section 4 and load {ALPHA158_PATH}."
    )

returns = df_processed[label_col].dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram with KDE
axes[0].hist(returns.clip(-0.3, 0.3), bins=80, color="#4C72B0", edgecolor="none", alpha=0.80, density=True)
returns.clip(-0.3, 0.3).plot.kde(ax=axes[0], color="#c44e52", linewidth=2, label="KDE")
axes[0].axvline(0, color="black", linestyle="--", linewidth=1, label="zero")
axes[0].axvline(returns.mean(), color="#dd8452", linestyle=":", linewidth=1.5, label=f"mean={returns.mean():.4f}")
axes[0].set_title(f"Distribution of {label_col}", fontweight="bold")
axes[0].set_xlabel("Forward Return")
axes[0].set_ylabel("Density")
axes[0].legend(fontsize=9)

# Per-ticker box plot (top 6)
sub = df_processed[df_processed["ticker"].isin(top_tickers)]
sub.boxplot(column=label_col, by="ticker", ax=axes[1], sym="", vert=True,
            medianprops=dict(color="red", linewidth=2))
axes[1].set_title("Forward Return by Ticker (top-volume sample)", fontweight="bold")
axes[1].set_xlabel("Ticker")
axes[1].set_ylabel("Return")
plt.suptitle("")

plt.tight_layout()
plt.show()

print(f"\n{label_col} summary statistics:\n")
print(returns.describe().to_string())

In [ ]:
# ── 3.4  Daily aggregate volume & mean price ─────────────────────────────────
daily_agg = df_raw.groupby("date").agg(
    total_volume=("volume", "sum"),
    mean_close=("close", "mean"),
    n_tickers=("ticker", "nunique"),
).sort_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].fill_between(daily_agg.index, daily_agg["total_volume"] / 1e6,
                     alpha=0.65, color="#DD8452")
axes[0].set_ylabel("Total Volume (M shares)")
axes[0].set_title("Aggregate Daily Volume Across Universe", fontweight="bold")

axes[1].plot(daily_agg["mean_close"], color="#4C72B0", linewidth=1.3)
axes[1].set_ylabel("Mean Close ($)")
axes[1].set_title("Mean Close Price Across Universe", fontweight="bold")
axes[1].set_xlabel("Date")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5  Pairwise return correlation heatmap ─────────────────────────────────
close_pivot = df_raw.pivot_table(index="date", columns="ticker", values="close")
ret_pivot   = close_pivot.pct_change().dropna()

# Correlation of the top-6 tickers
corr = ret_pivot[top_tickers].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.8, "label": "Pearson r (daily returns)"},
)
ax.set_title("Daily Return Correlation — top 6 tickers by volume", fontweight="bold")
plt.tight_layout()
plt.show()

# ── 3.6  Autocorrelation of label (leakage check) ────────────────────────────
sample_ticker = top_tickers[0]
label_series  = (
    df_processed[df_processed["ticker"] == sample_ticker]
    .set_index("date")
    .sort_index()[label_col]
)
pd.plotting.autocorrelation_plot(label_series, ax=plt.gca())
plt.title(f"Label Autocorrelation — {sample_ticker} ({label_col})", fontweight="bold")
plt.xlim(0, 30)
plt.tight_layout()
plt.show()

## 4  Qlib Dataset Build

This step builds two truthful artifacts from the processed OHLCV parquet: Qlib's daily binary provider layout and a materialized Alpha158 parquet. The Alpha158 parquet includes `LABEL0` as a 5-day forward return that is clipped and z-scored cross-sectionally per date. LightGBM and MASTER train on that artifact. StockMixer still trains directly from raw OHLCV.

In [ ]:
QLIB_CFG = {
    "qlib": {
        "provider_uri": str(QLIB_PROVIDER_URI),
        "market": "us_equities",
        "alpha158": {
            "output_path": str(ALPHA158_PATH),
            "label_column": "LABEL0",
            "label_horizon": 5,
            "normalize_label": True,
            "clip_quantiles": [0.025, 0.975],
            "vwap_mode": "typical_price",
        },
    },
    "data": {
        "processed_path": str(PROCESSED_PATH),
        "label_column": INGEST_LABEL_COL,
        "label_horizon": 5,
    },
    "paths": {"qlib_dir": str(DATA_DIR / "qlib")},
}

qlib_outputs = {}
try:
    qlib_outputs = build_qlib_dataset(QLIB_CFG)
    alpha_path = Path(qlib_outputs["alpha158"]).resolve()
    LGBM_CFG["data"]["processed_path"] = str(alpha_path)
    MASTER_CFG["data"]["processed_path"] = str(alpha_path)

    print("Qlib dataset built:")
    print(f"  provider_uri -> {qlib_outputs['provider_uri']}")
    print(f"  calendar     -> {qlib_outputs['calendar']}")
    print(f"  instruments  -> {qlib_outputs['instruments']}")
    print(f"  alpha158     -> {alpha_path}")
except Exception as exc:
    print(f"Qlib dataset build failed: {exc}")

## 5  MLflow Experiment Setup

Each model training run and the backtest comparison are logged as separate MLflow runs under the `stock_manager_pipeline` experiment. Launch the UI with the printed command.

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment          : {EXPERIMENT_NAME}")
print(f"\nLaunch MLflow UI:\n  mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}\n  → open http://127.0.0.1:5000")


def _require_alpha158_artifact(path: str | Path, label_column: str = "LABEL0") -> Path:
    path = Path(path)
    if not path.exists():
        raise ValueError(
            f"Alpha158 parquet not found at {path}. Run Section 4 before training LightGBM or MASTER."
        )

    columns = pd.read_parquet(path, columns=None).columns
    if label_column not in columns:
        raise ValueError(
            f"Alpha158 parquet at {path} is missing '{label_column}'. Re-run Section 4 to rebuild it with the current pipeline."
        )
    return path


def _log_run(model_name: str, cfg: dict, outputs: dict) -> str:
    """Log a training run to MLflow; returns the run_id."""
    with mlflow.start_run(run_name=model_name) as run:
        # ── parameters
        params = {
            "model_type":   cfg["model"]["type"],
            "train_end":    cfg["splits"]["train_end"],
            "valid_end":    cfg["splits"]["valid_end"],
            "test_end":     cfg["splits"].get("test_end", ""),
            "label_column": cfg["data"].get("label_column", ""),
        }
        params.update({f"hp_{k}": str(v) for k, v in cfg["model"].get("params", {}).items()})
        mlflow.log_params(params)

        # ── metrics
        metrics_path = outputs.get("metrics")
        if metrics_path and Path(str(metrics_path)).exists():
            with open(metrics_path) as f:
                mlflow.log_metrics(json.load(f))

        # ── artifacts
        for key, path in outputs.items():
            if path and Path(str(path)).exists():
                mlflow.log_artifact(str(path), artifact_path=key)

        run_id = run.info.run_id
    print(f"  ✓ MLflow run logged  [{model_name}]  run_id={run_id}")
    return run_id

## 6  Model Training

All three models are trained in sequence and logged to MLflow. LightGBM and MASTER consume the Alpha158 parquet built in Section 4. StockMixer trains directly on raw OHLCV. Failed models are skipped so the rest of the notebook can continue.

In [ ]:
print("Training LightGBM Alpha158 …")
lgbm_outputs  = {}
lgbm_metrics  = {}
lgbm_run_id   = None

try:
    _require_alpha158_artifact(LGBM_CFG["data"]["processed_path"], LGBM_CFG["data"]["label_column"])
    lgbm_outputs = train_from_config(LGBM_CFG)
    with open(lgbm_outputs["metrics"]) as f:
        lgbm_metrics = json.load(f)
    lgbm_run_id = _log_run("lightgbm_alpha158", LGBM_CFG, lgbm_outputs)

    print(f"\n  Rank IC  : {lgbm_metrics.get('rank_ic', 'N/A'):.4f}")
    print(f"  MSE      : {lgbm_metrics.get('mse', 'N/A'):.6f}")
    print(f"  Artifacts: {list(lgbm_outputs.keys())}")
except Exception as exc:
    print(f"LightGBM training failed: {exc}")

### 6.1a  LightGBM — Hyperparameter Optimisation (Optuna)

Bayesian search over `n_estimators`, `learning_rate`, `num_leaves`, `min_child_samples`, `subsample`, `colsample_bytree`, and `reg_lambda`. Objective: maximise **Rank IC** on the validation split. Runs 30 Optuna trials (TPE sampler).


In [ ]:
import math as _math
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

from stock_manager.backtest.metrics import rank_ic as _rank_ic

# ── guard: need alpha158 artifact ─────────────────────────────────────────────
_alpha_path = LGBM_CFG["data"]["processed_path"]
try:
    _require_alpha158_artifact(_alpha_path, LGBM_CFG["data"]["label_column"])
    _run_hpo = True
except Exception as _e:
    print(f"⚠  Alpha158 artifact missing — skipping LightGBM HPO\n   ({_e})")
    _run_hpo = False

best_lgbm_params  = {}
best_lgbm_val_ic  = float("nan")

if _run_hpo:
    import lightgbm as lgb

    _frame = pd.read_parquet(_alpha_path)
    _frame["date"] = pd.to_datetime(_frame["date"]).dt.tz_localize(None)
    _label = LGBM_CFG["data"]["label_column"]
    _frame = _frame.dropna(subset=[_label]).sort_values(["date", "ticker"]).reset_index(drop=True)
    _feat_cols = [
        c for c in _frame.columns
        if c not in {"date", "ticker", _label} and pd.api.types.is_numeric_dtype(_frame[c])
    ]

    _train_hpo = _frame[_frame["date"] <= pd.Timestamp(SPLITS["train_end"])]
    _valid_hpo = _frame[
        (_frame["date"] > pd.Timestamp(SPLITS["train_end"])) &
        (_frame["date"] <= pd.Timestamp(SPLITS["valid_end"]))
    ]

    def _lgbm_objective(trial: optuna.Trial) -> float:
        p = {
            "n_estimators":      trial.suggest_int("n_estimators", 100, 600, step=50),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 31, 127, step=16),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 60, step=10),
            "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
            "n_jobs": -1,
            "verbosity": -1,
        }
        m = lgb.LGBMRegressor(**p)
        m.fit(
            _train_hpo[_feat_cols], _train_hpo[_label],
            eval_set=[(_valid_hpo[_feat_cols], _valid_hpo[_label])],
            eval_metric="l2",
            callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
        )
        pred_df = pd.DataFrame({
            "prediction": m.predict(_valid_hpo[_feat_cols]),
            "label":  _valid_hpo[_label].values,
            "date":   _valid_hpo["date"].values,
            "ticker": _valid_hpo["ticker"].values,
        })
        ic = float(_rank_ic(pred_df))
        return ic if not _math.isnan(ic) else -999.0

    HPO_N_TRIALS = 30
    print(f"Running Optuna LightGBM HPO — {HPO_N_TRIALS} trials …")
    lgbm_study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    lgbm_study.optimize(_lgbm_objective, n_trials=HPO_N_TRIALS, show_progress_bar=True)

    best_lgbm_params = lgbm_study.best_params
    best_lgbm_val_ic = lgbm_study.best_value
    print(f"\n✓  Best validation Rank IC : {best_lgbm_val_ic:.4f}")
    print(f"   Best params             : {best_lgbm_params}")

    # ── Plot: optimisation history ────────────────────────────────────────────
    trial_values = [t.value for t in lgbm_study.trials if t.value is not None and not _math.isnan(t.value)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(trial_values, marker="o", markersize=4, linewidth=1, color="#4C72B0", alpha=0.7)
    running_max = [max(trial_values[:i+1]) for i in range(len(trial_values))]
    axes[0].plot(running_max, color="#c44e52", linewidth=1.8, linestyle="--", label=f"best={best_lgbm_val_ic:.4f}")
    axes[0].set_title("LightGBM HPO — Rank IC per Trial", fontweight="bold")
    axes[0].set_xlabel("Trial"); axes[0].set_ylabel("Validation Rank IC")
    axes[0].legend(fontsize=9)

    param_names = list(best_lgbm_params.keys())
    param_vals  = [float(best_lgbm_params[k]) for k in param_names]
    axes[1].barh(param_names, param_vals, color="#4C72B0", alpha=0.8, edgecolor="none")
    axes[1].set_title("Best Hyperparameter Values", fontweight="bold")
    axes[1].set_xlabel("Value")
    plt.tight_layout(); plt.show()


### 6.1b  LightGBM — Re-train with Best HPO Params

Retrain with the winner config and compare test-set Rank IC against the baseline.


In [ ]:
lgbm_hpo_outputs = {}
lgbm_hpo_metrics = {}

if best_lgbm_params:
    import copy as _copy
    _hpo_cfg = _copy.deepcopy(LGBM_CFG)
    _hpo_cfg["model"]["params"] = {
        **best_lgbm_params,
        "early_stopping_rounds": 20,
        "n_jobs": -1,
    }
    _hpo_cfg["paths"]["report_dir"] = str(REPORT_DIR / "lightgbm_alpha158_hpo")

    print("Re-training LightGBM with best HPO params …")
    try:
        lgbm_hpo_outputs = train_from_config(_hpo_cfg)
        with open(lgbm_hpo_outputs["metrics"]) as f:
            lgbm_hpo_metrics = json.load(f)
        _log_run("lightgbm_alpha158_hpo", _hpo_cfg, lgbm_hpo_outputs)

        baseline_ic = lgbm_metrics.get("rank_ic", float("nan"))
        hpo_ic      = lgbm_hpo_metrics.get("rank_ic", float("nan"))
        delta       = hpo_ic - baseline_ic if not (_math.isnan(hpo_ic) or _math.isnan(baseline_ic)) else float("nan")

        print(f"\n  Baseline Rank IC : {baseline_ic:.4f}")
        print(f"  HPO Rank IC      : {hpo_ic:.4f}  (Δ {delta:+.4f})")
        print(f"  MSE              : {lgbm_hpo_metrics.get('mse', float('nan')):.6f}")

        # ── side-by-side bar ──────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(7, 4))
        labels   = ["Baseline", "HPO"]
        ic_vals  = [baseline_ic, hpo_ic]
        colors   = ["#4C72B0", "#2ca02c" if delta >= 0 else "#d62728"]
        bars = ax.bar(labels, ic_vals, color=colors, alpha=0.85, edgecolor="none", width=0.4)
        ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=11, fontweight="bold")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title("LightGBM: Baseline vs HPO — Test Rank IC", fontweight="bold")
        ax.set_ylabel("Rank IC")
        plt.tight_layout(); plt.show()

    except Exception as exc:
        print(f"HPO retrain failed: {exc}")
else:
    print("No HPO params found — run the HPO cell first.")


### 6.2  MASTER  *(requires PyTorch)*

This path uses the upstream MASTER architecture with a local market-information adapter built from equal-weight, dollar-volume-weighted, and median universe summaries.

In [ ]:
print("Training MASTER …")
master_outputs = {}
master_metrics = {}
master_run_id  = None

try:
    _require_alpha158_artifact(MASTER_CFG["data"]["processed_path"], MASTER_CFG["data"]["label_column"])
    master_outputs = train_from_config(MASTER_CFG)
    with open(master_outputs["metrics"]) as f:
        master_metrics = json.load(f)
    master_run_id = _log_run("master", MASTER_CFG, master_outputs)

    print(f"\n  Rank IC  : {master_metrics.get('rank_ic', 'N/A'):.4f}")
    print(f"  Artifacts: {list(master_outputs.keys())}")
except Exception as exc:
    print(f"MASTER training failed: {exc}")

### 6.3  StockMixer  *(requires PyTorch)*

This path uses the upstream StockMixer architecture and native ranking loss on the raw OHLCV parquet.

In [ ]:
print("Training StockMixer …")
stockmixer_outputs = {}
stockmixer_metrics = {}
stockmixer_run_id  = None

try:
    stockmixer_outputs = train_from_config(STOCKMIXER_CFG)
    with open(stockmixer_outputs["metrics"]) as f:
        stockmixer_metrics = json.load(f)
    stockmixer_run_id = _log_run("stockmixer", STOCKMIXER_CFG, stockmixer_outputs)

    print(f"\n  Rank IC  : {stockmixer_metrics.get('rank_ic', 'N/A'):.4f}")
    print(f"  Artifacts: {list(stockmixer_outputs.keys())}")
except Exception as exc:
    print(f"StockMixer training failed: {exc}")

### 6.4  Optimizer Sweep — MASTER & StockMixer

Compares **Adam**, **AdamW**, and **RMSprop** for both PyTorch models.  
Each variant runs with a reduced epoch budget for speed; the best optimizer per model is highlighted.  
The full configs (`MASTER_CFG` / `STOCKMIXER_CFG`) already default to the recommended settings (AdamW for MASTER, Adam for StockMixer).


In [ ]:
import copy as _copy, json as _json, math as _math
from pathlib import Path as _Path

def _np_isnan(x: float) -> bool:
    try: return _math.isnan(float(x))
    except: return True

# ── Sweep settings ────────────────────────────────────────────────────────────
OPTIMIZER_SWEEP        = ["adam", "adamw", "rmsprop"]
SWEEP_MASTER_EPOCHS    = 10    # raise to 40 for full training
SWEEP_STOCKMIXER_EPOCHS = 20   # raise to 100 for full training

optimizer_sweep_results: list[dict] = []

def _sweep_model(base_cfg: dict, model_label: str, opt_name: str, extra_params: dict) -> dict:
    """Train one optimizer variant; returns a results dict."""
    cfg = _copy.deepcopy(base_cfg)
    cfg["model"]["params"].update({"optimizer": opt_name, **extra_params})
    cfg["paths"]["report_dir"] = str(REPORT_DIR / f"{model_label}_opt_{opt_name}")
    try:
        outputs = train_from_config(cfg)
        with open(outputs["metrics"]) as fh:
            metrics = _json.load(fh)
        _log_run(f"{model_label}_{opt_name}", cfg, outputs)
        return {"model": model_label, "optimizer": opt_name,
                "rank_ic": metrics.get("rank_ic", float("nan")),
                "mse":     metrics.get("mse",     float("nan")),
                "outputs": outputs}
    except Exception as exc:
        print(f"  ✗ {model_label}/{opt_name}: {exc}")
        return {"model": model_label, "optimizer": opt_name,
                "rank_ic": float("nan"), "mse": float("nan"), "outputs": {}}


# ── MASTER sweep ──────────────────────────────────────────────────────────────
master_alpha_ok = _Path(MASTER_CFG["data"]["processed_path"]).exists()
if master_alpha_ok:
    for opt in OPTIMIZER_SWEEP:
        print(f"MASTER ▶ optimizer={opt} …")
        result = _sweep_model(
            MASTER_CFG, "master", opt,
            extra_params={
                "n_epochs": SWEEP_MASTER_EPOCHS,
                "weight_decay": 1e-2 if opt == "adamw" else 0.0,
                "grad_clip": 3.0,
            },
        )
        optimizer_sweep_results.append(result)
        if not _np_isnan(result["rank_ic"]):
            print(f"         Rank IC={result['rank_ic']:.4f}  MSE={result['mse']:.6f}")
else:
    print("⚠  MASTER alpha158 path not found — skipping MASTER optimizer sweep")

# ── StockMixer sweep ──────────────────────────────────────────────────────────
stockmixer_data_ok = _Path(STOCKMIXER_CFG["data"]["processed_path"]).exists()
if stockmixer_data_ok:
    for opt in OPTIMIZER_SWEEP:
        print(f"StockMixer ▶ optimizer={opt} …")
        result = _sweep_model(
            STOCKMIXER_CFG, "stockmixer", opt,
            extra_params={
                "epochs": SWEEP_STOCKMIXER_EPOCHS,
                "weight_decay": 1e-3 if opt == "adamw" else 0.0,
                "grad_clip": 5.0,
            },
        )
        optimizer_sweep_results.append(result)
        if not _np_isnan(result["rank_ic"]):
            print(f"         Rank IC={result['rank_ic']:.4f}  MSE={result['mse']:.6f}")
else:
    print("⚠  StockMixer OHLCV data not found — skipping StockMixer optimizer sweep")

# ── Summary ───────────────────────────────────────────────────────────────────
sweep_rows = [r for r in optimizer_sweep_results if not _np_isnan(r["rank_ic"])]
if sweep_rows:
    sweep_df = pd.DataFrame([
        {"Model": r["model"], "Optimizer": r["optimizer"],
         "Rank IC": r["rank_ic"], "MSE": r["mse"]}
        for r in sweep_rows
    ])
    print("\n" + "=" * 55)
    print(" OPTIMIZER SWEEP RESULTS")
    print("=" * 55)
    print(sweep_df.to_string(index=False, float_format="{:.5f}".format))

    # grouped bar chart
    model_names = list(sweep_df["Model"].unique())
    palette     = sns.color_palette("muted", n_colors=len(OPTIMIZER_SWEEP))
    opt_color   = dict(zip(OPTIMIZER_SWEEP, palette))
    fig, axes   = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric in zip(axes, ["Rank IC", "MSE"]):
        x = np.arange(len(model_names))
        w = 0.25
        for i, opt in enumerate(OPTIMIZER_SWEEP):
            sub  = sweep_df[sweep_df["Optimizer"] == opt].set_index("Model")
            vals = [sub.loc[m, metric] if m in sub.index else float("nan") for m in model_names]
            bars = ax.bar(x + i * w, vals, width=w, label=opt,
                          color=opt_color[opt], alpha=0.85, edgecolor="none")
            ax.bar_label(bars, fmt="%.4f", padding=2, fontsize=8)
        ax.set_xticks(x + w); ax.set_xticklabels(model_names)
        ax.set_title(f"{metric} by Model & Optimizer", fontweight="bold")
        ax.set_ylabel(metric)
        ax.legend(title="Optimizer", fontsize=9)

    plt.tight_layout(); plt.show()

    print("\nBest optimizer per model (by Rank IC):")
    for mn in model_names:
        sub = sweep_df[sweep_df["Model"] == mn].sort_values("Rank IC", ascending=False)
        if not sub.empty:
            best_row = sub.iloc[0]
            print(f"  {mn:20s}  →  {best_row['Optimizer']}  (Rank IC={best_row['Rank IC']:.4f})")
else:
    print("No sweep results — ensure training data is available and re-run this cell.")


## 7  Post-training Visualisation

Analyse what each model learned: feature importance, rolling IC over time, and prediction quality scatter.

In [ ]:
# ── 7.1  LightGBM feature importance ────────────────────────────────────────
lgbm_model_path = lgbm_outputs.get("model")

if lgbm_model_path and Path(str(lgbm_model_path)).exists():
    with open(lgbm_model_path, "rb") as f:
        lgbm_model = pickle.load(f)

    importance = (
        pd.Series(lgbm_model.feature_importances_, index=lgbm_model.feature_name_)
        .sort_values(ascending=True)
        .tail(25)
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette("Blues_r", n_colors=len(importance))
    importance.plot.barh(ax=ax, color=colors[::-1], edgecolor="none")
    ax.set_title("LightGBM — Top 25 Features by Gain Importance", fontweight="bold")
    ax.set_xlabel("Feature Importance (gain)")
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout()
    plt.show()

    print(f"Total features: {len(lgbm_model.feature_importances_)}")
    print(f"Non-zero features: {(lgbm_model.feature_importances_ > 0).sum()}")
else:
    print("LightGBM model not available — skipping feature importance.")

In [ ]:
# ── 7.2  Rolling daily Rank IC — all models ──────────────────────────────────
all_pred_paths = {
    "LightGBM":   lgbm_outputs.get("predictions"),
    "MASTER":     master_outputs.get("predictions"),
    "StockMixer": stockmixer_outputs.get("predictions"),
}

available_preds = {
    name: path
    for name, path in all_pred_paths.items()
    if path and Path(str(path)).exists()
}

if available_preds:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for model_name, pred_path in available_preds.items():
        preds = pd.read_parquet(pred_path)
        preds["date"] = pd.to_datetime(preds["date"])

        daily_ic = preds.groupby("date")[["prediction", "label"]].apply(
            lambda d: d.corr(method="spearman").iloc[0, 1]
        ).dropna()

        rolling_ic = daily_ic.rolling(20, min_periods=5).mean()
        axes[0].plot(rolling_ic, label=f"{model_name} (mean={daily_ic.mean():.3f})", linewidth=1.6)

    axes[0].axhline(0, color="black", linestyle="--", linewidth=0.9)
    axes[0].fill_between(rolling_ic.index, 0, rolling_ic,
                         where=(rolling_ic > 0), alpha=0.08, color="green")
    axes[0].fill_between(rolling_ic.index, 0, rolling_ic,
                         where=(rolling_ic < 0), alpha=0.08, color="red")
    axes[0].set_title("Rolling 20-day Rank IC", fontweight="bold")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Rank IC")
    axes[0].legend(fontsize=9)

    # ── Prediction vs actual scatter (LightGBM sample) ──────────────────
    lgbm_path = lgbm_outputs.get("predictions")
    if lgbm_path and Path(str(lgbm_path)).exists():
        preds_all = pd.read_parquet(lgbm_path)
        preds_sample = preds_all.sample(min(3000, len(preds_all)), random_state=42)
        axes[1].scatter(preds_sample["prediction"], preds_sample["label"],
                        alpha=0.18, s=5, color="#4C72B0", rasterized=True)
        # regression line
        m, b = np.polyfit(preds_sample["prediction"], preds_sample["label"], 1)
        x_line = np.linspace(preds_sample["prediction"].min(), preds_sample["prediction"].max(), 100)
        axes[1].plot(x_line, m * x_line + b, color="#c44e52", linewidth=1.8, label="OLS fit")
        rho = preds_sample[["prediction", "label"]].corr(method="spearman").iloc[0, 1]
        axes[1].text(0.05, 0.95, f"Spearman ρ = {rho:.3f}",
                     transform=axes[1].transAxes, fontsize=10, verticalalignment="top",
                     bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.7))
        axes[1].set_title("LightGBM: Predicted vs Actual Return (test set)", fontweight="bold")
        axes[1].set_xlabel("Predicted Return")
        axes[1].set_ylabel("Actual Return")
        axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("No prediction artifacts available for plotting.")

In [ ]:
# ── 7.3  Training metrics summary table ──────────────────────────────────────
trained_models = {
    "LightGBM":   lgbm_metrics,
    "MASTER":     master_metrics,
    "StockMixer": stockmixer_metrics,
}
rows = [
    {"Model": name, **metrics}
    for name, metrics in trained_models.items()
    if metrics
]

if rows:
    summary_df = pd.DataFrame(rows).set_index("Model")
    print("=" * 60)
    print(" TRAINING METRICS SUMMARY (test set)")
    print("=" * 60)
    print(summary_df.to_string(float_format="{:.5f}".format))
    print("=" * 60)

    # Bar chart of Rank IC per model
    if "rank_ic" in summary_df.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ["#2ca02c" if v > 0 else "#d62728" for v in summary_df["rank_ic"]]
        summary_df["rank_ic"].plot.bar(ax=ax, color=colors, edgecolor="none", alpha=0.85)
        ax.bar_label(ax.containers[0], fmt="%.4f", padding=3, fontsize=10)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title("Rank IC per Model (test set)", fontweight="bold")
        ax.set_ylabel("Rank IC")
        ax.tick_params(axis="x", rotation=0)
        plt.tight_layout()
        plt.show()
else:
    print("No training metrics available yet.")

## 8  Backtesting & Model Comparison

Runs a long-short backtest over the test period for each model at 0 / 5 / 10 bps transaction cost. Results are saved to CSV/JSON and logged to MLflow.

In [ ]:
print("Running backtest comparison …")

# Only include models whose prediction files exist
available_bt_models = [
    m for m in BACKTEST_CFG["models"]
    if Path(m["predictions_path"]).exists()
]
skipped = set(m["name"] for m in BACKTEST_CFG["models"]) - {m["name"] for m in available_bt_models}
for name in skipped:
    print(f"  ⚠  Skipping {name} — predictions not found")

comp_df = pd.DataFrame()
bt_outputs = {}

if available_bt_models:
    bt_cfg = {**BACKTEST_CFG, "models": available_bt_models}
    bt_outputs = compare_models(bt_cfg)

    comp_df = pd.read_csv(bt_outputs["csv"])
    print("\nModel Comparison (0 bps cost):")
    print(comp_df[comp_df["cost_bps"] == 0].to_string(index=False))

    # ── Log comparison run to MLflow ──────────────────────────────────────
    with mlflow.start_run(run_name="backtest_comparison"):
        for _, row in comp_df[comp_df["cost_bps"] == 0].iterrows():
            prefix = row["model"].replace("-", "_")
            mlflow.log_metrics({
                f"{prefix}_rank_ic":          float(row.get("rank_ic", 0)),
                f"{prefix}_sharpe":           float(row.get("sharpe", 0)),
                f"{prefix}_annualized_return": float(row.get("annualized_return", 0)),
                f"{prefix}_max_drawdown":     float(row.get("max_drawdown", 0)),
                f"{prefix}_hit_rate":         float(row.get("hit_rate", 0)),
            })
        mlflow.log_artifact(str(bt_outputs["csv"]))
        mlflow.log_artifact(str(bt_outputs["json"]))
        mlflow.log_artifact(str(bt_outputs["summary"]))
    print("\n  ✓ Backtest comparison logged to MLflow")
else:
    print("No models available for backtest.")

## 9  Post-backtest Visualisation

Equity curves, drawdown analysis, performance comparison bar charts, cost-sensitivity, and a final summary table.

In [ ]:
# ── 9.1  Long-short equity curves ────────────────────────────────────────────
if available_preds:
    fig, ax = plt.subplots(figsize=(14, 6))

    for model_name, pred_path in available_preds.items():
        preds = pd.read_parquet(pred_path)
        preds["date"] = pd.to_datetime(preds["date"])
        ls_ret = long_short_returns(preds, top_quantile=0.2)
        equity = (1 + ls_ret.fillna(0)).cumprod()
        final_val = equity.iloc[-1]
        ax.plot(equity, linewidth=1.8, label=f"{model_name}  (final={final_val:.2f}x)")

    ax.axhline(1.0, color="black", linestyle="--", linewidth=0.9, alpha=0.6, label="Baseline")
    ax.set_title("Long-Short Equity Curves  (top/bottom 20% quantile, 0 bps cost)",
                 fontweight="bold")
    ax.set_ylabel("Portfolio Value  (start = 1.0)")
    ax.set_xlabel("Date")
    ax.legend(fontsize=10)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f"))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("No prediction artifacts available — run Section 6 first.")

In [ ]:
# ── 9.2  Drawdown chart ───────────────────────────────────────────────────────
if available_preds:
    fig, ax = plt.subplots(figsize=(14, 5))
    palette = sns.color_palette("muted", n_colors=len(available_preds))

    for (model_name, pred_path), color in zip(available_preds.items(), palette):
        preds = pd.read_parquet(pred_path)
        preds["date"] = pd.to_datetime(preds["date"])
        ls_ret = long_short_returns(preds, top_quantile=0.2)
        equity = (1 + ls_ret.fillna(0)).cumprod()
        dd = equity / equity.cummax() - 1
        ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color=color)
        ax.plot(dd, linewidth=1.2, color=color,
                label=f"{model_name}  (max={dd.min():.2%})")

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Long-Short Drawdown", fontweight="bold")
    ax.set_ylabel("Drawdown")
    ax.set_xlabel("Date")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(fontsize=10)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 9.3  Performance comparison bar charts (0 bps) ────────────────────────────
if not comp_df.empty:
    df0 = comp_df[comp_df["cost_bps"] == 0].set_index("model")

    plot_metrics = [
        ("rank_ic",           "Rank IC"),
        ("sharpe",            "Annualised Sharpe"),
        ("annualized_return", "Annualised Return"),
        ("max_drawdown",      "Max Drawdown"),
        ("hit_rate",          "Hit Rate"),
        ("icir",              "IC IR"),
    ]
    available_metrics = [(col, title) for col, title in plot_metrics if col in df0.columns]
    ncols = min(3, len(available_metrics))
    nrows = (len(available_metrics) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows))
    axes = np.array(axes).flatten() if nrows * ncols > 1 else [axes]

    for ax, (col, title) in zip(axes, available_metrics):
        vals = df0[col]
        colors = ["#2ca02c" if v >= 0 else "#d62728" for v in vals]
        bars = ax.bar(df0.index, vals, color=colors, edgecolor="none", alpha=0.85)
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(title, fontweight="bold")
        ax.tick_params(axis="x", rotation=15)

    # hide any unused axes
    for ax in axes[len(available_metrics):]:
        ax.set_visible(False)

    fig.suptitle("Model Performance Summary  (0 bps transaction cost)",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No comparison data available — run Section 8 first.")

In [ ]:
# ── 9.4  Cost sensitivity — Sharpe vs transaction cost ───────────────────────
if not comp_df.empty and "sharpe" in comp_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Sharpe vs cost
    for model_name, grp in comp_df.groupby("model"):
        axes[0].plot(grp["cost_bps"], grp["sharpe"], marker="o", linewidth=2, label=model_name)
    axes[0].axhline(0, color="black", linestyle="--", linewidth=0.8)
    axes[0].set_title("Sharpe Ratio vs Transaction Cost", fontweight="bold")
    axes[0].set_xlabel("Cost (bps)")
    axes[0].set_ylabel("Sharpe Ratio")
    axes[0].legend()

    # Annualised return vs cost
    if "annualized_return" in comp_df.columns:
        for model_name, grp in comp_df.groupby("model"):
            axes[1].plot(grp["cost_bps"], grp["annualized_return"],
                         marker="s", linewidth=2, label=model_name)
        axes[1].axhline(0, color="black", linestyle="--", linewidth=0.8)
        axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        axes[1].set_title("Annualised Return vs Transaction Cost", fontweight="bold")
        axes[1].set_xlabel("Cost (bps)")
        axes[1].set_ylabel("Annualised Return")
        axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# ── 9.5  Final summary table ─────────────────────────────────────────────────
display_cols = ["model", "rank_ic", "ic", "icir", "sharpe",
                "annualized_return", "max_drawdown", "hit_rate", "long_short_spread"]

if not comp_df.empty:
    summary = comp_df[comp_df["cost_bps"] == 0][
        [c for c in display_cols if c in comp_df.columns]
    ].copy()

    best_model = (
        summary.sort_values("rank_ic", ascending=False).iloc[0]["model"]
        if "rank_ic" in summary.columns else "N/A"
    )

    print("=" * 80)
    print(" FINAL PERFORMANCE SUMMARY  (0 bps transaction cost, long-short backtest)")
    print("=" * 80)
    float_cols = summary.select_dtypes("float").columns
    for col in float_cols:
        summary[col] = summary[col].apply(lambda x: f"{x:.5f}")
    print(summary.to_string(index=False))
    print("=" * 80)
    print(f"\n  Best model by Rank IC : {best_model}")
    print(f"  MLflow experiment     : {EXPERIMENT_NAME}")
    print(f"  Tracking URI          : {MLFLOW_TRACKING_URI}")
    print("\n⚠  Research-only output. Past performance does not guarantee future results.")
else:
    print("Run Section 8 to generate backtest comparison data.")